# Can we predict 30-day readmission?

This notebook evaluates whether the supplied encounter-level variables can predict `readmit_30d`.

**Short answer:** there is predictive signal, but it is modest. In patient-grouped cross-validation, the strongest tested model reaches ROC AUC around **0.63** and average precision around **0.30**, compared with a readmission prevalence near **0.20**. That may support risk stratification, but it is not strong enough to justify autonomous clinical decisions. External and temporal validation are required before use.

The analysis deliberately:

- excludes `person_key` from predictors;
- treats `postcode_area` as categorical rather than continuous;
- keeps all encounters from a person in the same fold to prevent patient leakage;
- compares against a prevalence-only baseline;
- reports discrimination, calibration, uncertainty, and threshold tradeoffs;
- does not claim causality or clinical utility from retrospective discrimination alone.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    precision_recall_curve, roc_auc_score, roc_curve
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## 1. Load the data and dictionary

The file paths are relative so the notebook works when opened from this directory.

In [ ]:
DATA_PATH = Path("data.csv")
DICTIONARY_PATH = Path("columns.csv")

data = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)

display(dictionary)
display(data.head())
print(f"Rows: {len(data):,}; columns: {data.shape[1]}; people: {data['person_key'].nunique():,}")

## 2. Data-quality and leakage audit

`person_key` is an identifier, not a clinical predictor. It is useful only for grouping repeated encounters during validation. The outcome occurs after discharge; all other fields are described as available no later than discharge. A true production audit should still confirm how every field is timestamped and generated.

In [ ]:
TARGET = "readmit_30d"
ID = "person_key"

expected = set(dictionary["column"])
assert expected == set(data.columns), "Data and dictionary columns do not match."
assert set(data[TARGET].dropna().unique()).issubset({0, 1}), "Target must be binary."

audit = pd.DataFrame({
    "dtype": data.dtypes.astype(str),
    "missing_n": data.isna().sum(),
    "missing_pct": data.isna().mean().mul(100),
    "unique_n": data.nunique(dropna=False),
})
display(audit)

print(f"Exact duplicate rows: {data.duplicated().sum():,}")
print(f"Duplicate person/visit pairs: {data.duplicated([ID, 'visit_number']).sum():,}")
visits_per_person = data.groupby(ID).size()
print(f"People with >1 encounter: {(visits_per_person > 1).sum():,} / {len(visits_per_person):,}")
print(f"Maximum encounters for one person: {visits_per_person.max():,}")

In [ ]:
prevalence = data[TARGET].mean()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.countplot(data=data, x=TARGET, ax=axes[0], color="#4C78A8")
axes[0].set(title=f"Outcome balance (prevalence = {prevalence:.1%})", xlabel="Readmitted within 30 days")

sns.histplot(visits_per_person, discrete=True, ax=axes[1], color="#F58518")
axes[1].set(title="Repeated encounters require grouped validation", xlabel="Encounters per person")
plt.tight_layout()
plt.show()

In [ ]:
numeric_summary = data.drop(columns=[ID]).describe().T
display(numeric_summary)

numeric_features_for_plot = [
    "visit_number", "birth_year", "bmi", "length_of_stay_days",
    "n_medications", "lab_a", "lab_b"
]
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, feature in zip(axes.flat, numeric_features_for_plot):
    sns.histplot(data=data, x=feature, hue=TARGET, stat="density", common_norm=False,
                 element="step", fill=False, ax=ax)
    ax.set_title(feature)
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## 3. Modeling design

The intended validation question is: **how well does the model generalize to encounters from people not used in training?** We therefore use five-fold `StratifiedGroupKFold`, stratifying approximately by outcome while assigning a person to one fold only. This is stricter and safer than a random row split.

Exact age cannot be calculated because no encounter date is supplied, so `birth_year` is retained as an age proxy. `postcode_area` is encoded as a category; its use may introduce geographic or socioeconomic inequity and should be reviewed before deployment.

Metrics:

- **ROC AUC:** ranking ability across thresholds; 0.5 is random.
- **Average precision (AP):** especially informative for the minority outcome; the no-skill reference is prevalence.
- **Brier score:** squared probability error; lower is better.

Accuracy at a 0.5 cutoff is intentionally not the headline metric: predicting every encounter as “no readmission” would already be about 80% accurate.

In [ ]:
features = [c for c in data.columns if c not in [TARGET, ID]]
categorical_features = ["sex", "postcode_area"]
numeric_features = [c for c in features if c not in categorical_features]

X = data[features].copy()
y = data[TARGET].astype(int).copy()
groups = data[ID].copy()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("categorical", categorical_pipe, categorical_features),
])

models = {
    "Prevalence baseline": DummyClassifier(strategy="prior"),
    "Logistic regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)),
    ]),
    "Gradient boosting": Pipeline([
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.05, max_iter=200, l2_regularization=1.0,
            random_state=RANDOM_STATE
        )),
    ]),
    "Random forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=10, n_jobs=-1,
            random_state=RANDOM_STATE
        )),
    ]),
}

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(cv.split(X, y, groups))
for train_idx, test_idx in splits:
    assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))

fold_summary = pd.DataFrame([
    {
        "fold": fold + 1,
        "train_rows": len(train_idx),
        "validation_rows": len(test_idx),
        "validation_people": groups.iloc[test_idx].nunique(),
        "validation_prevalence": y.iloc[test_idx].mean(),
    }
    for fold, (train_idx, test_idx) in enumerate(splits)
])
display(fold_summary)

## 4. Patient-grouped cross-validation

Every row receives an out-of-fold probability from a model that was not trained on that person. Fold means and standard deviations show stability across patient groups. No hyperparameter search is performed, avoiding another layer of selection optimism on this small benchmark.

In [ ]:
oof_predictions = {}
fold_records = []

for model_name, model in models.items():
    oof = np.full(len(data), np.nan)
    for fold, (train_idx, test_idx) in enumerate(splits, start=1):
        fitted = clone(model).fit(X.iloc[train_idx], y.iloc[train_idx])
        probability = fitted.predict_proba(X.iloc[test_idx])[:, 1]
        oof[test_idx] = probability
        fold_records.append({
            "model": model_name,
            "fold": fold,
            "roc_auc": roc_auc_score(y.iloc[test_idx], probability),
            "average_precision": average_precision_score(y.iloc[test_idx], probability),
            "brier_score": brier_score_loss(y.iloc[test_idx], probability),
        })
    assert not np.isnan(oof).any()
    oof_predictions[model_name] = oof

fold_metrics = pd.DataFrame(fold_records)
summary = fold_metrics.groupby("model").agg(
    roc_auc_mean=("roc_auc", "mean"),
    roc_auc_sd=("roc_auc", "std"),
    average_precision_mean=("average_precision", "mean"),
    average_precision_sd=("average_precision", "std"),
    brier_mean=("brier_score", "mean"),
    brier_sd=("brier_score", "std"),
).sort_values("roc_auc_mean", ascending=False)
display(summary.style.format("{:.3f}"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for model_name, probability in oof_predictions.items():
    fpr, tpr, _ = roc_curve(y, probability)
    axes[0].plot(fpr, tpr, label=f"{model_name} ({roc_auc_score(y, probability):.3f})")
    precision, recall, _ = precision_recall_curve(y, probability)
    axes[1].plot(recall, precision, label=f"{model_name} ({average_precision_score(y, probability):.3f})")

axes[0].plot([0, 1], [0, 1], "--", color="grey", linewidth=1)
axes[0].set(title="Out-of-fold ROC curves", xlabel="False-positive rate", ylabel="True-positive rate")
axes[1].axhline(prevalence, linestyle="--", color="grey", linewidth=1, label=f"Prevalence ({prevalence:.3f})")
axes[1].set(title="Out-of-fold precision–recall curves", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Uncertainty for the selected benchmark

The following percentile intervals resample **people**, not rows, preserving within-person dependence. They describe sampling uncertainty in these out-of-fold predictions; they do not cover dataset shift or the optimism from choosing the best of several models.

In [ ]:
best_model_name = summary.index[0]
best_oof = oof_predictions[best_model_name]

def clustered_bootstrap_metrics(y_true, probability, cluster, n_boot=500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    unique_clusters = np.asarray(pd.unique(cluster))
    cluster_to_indices = {g: np.flatnonzero(np.asarray(cluster) == g) for g in unique_clusters}
    estimates = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_clusters, size=len(unique_clusters), replace=True)
        idx = np.concatenate([cluster_to_indices[g] for g in sampled])
        if np.unique(np.asarray(y_true)[idx]).size < 2:
            continue
        estimates.append([
            roc_auc_score(np.asarray(y_true)[idx], np.asarray(probability)[idx]),
            average_precision_score(np.asarray(y_true)[idx], np.asarray(probability)[idx]),
            brier_score_loss(np.asarray(y_true)[idx], np.asarray(probability)[idx]),
        ])
    return np.asarray(estimates)

boot = clustered_bootstrap_metrics(y, best_oof, groups)
point = np.array([
    roc_auc_score(y, best_oof),
    average_precision_score(y, best_oof),
    brier_score_loss(y, best_oof),
])
ci = np.percentile(boot, [2.5, 97.5], axis=0)
uncertainty = pd.DataFrame({
    "metric": ["ROC AUC", "Average precision", "Brier score"],
    "estimate": point,
    "95% CI lower": ci[0],
    "95% CI upper": ci[1],
})
print(f"Selected benchmark: {best_model_name}")
display(uncertainty.style.format({c: "{:.3f}" for c in uncertainty.columns if c != "metric"}))

## 5. Threshold tradeoffs

A probability model does not determine a clinical action threshold. The table below illustrates tradeoffs using out-of-fold predictions. A threshold must ultimately reflect intervention benefit, capacity, false-positive cost, and false-negative harm, and should be selected in a separate training/tuning process—not from this report's validation results.

In [ ]:
def threshold_metrics(y_true, probability, thresholds):
    rows = []
    y_array = np.asarray(y_true)
    for threshold in thresholds:
        pred = probability >= threshold
        tn, fp, fn, tp = confusion_matrix(y_array, pred, labels=[0, 1]).ravel()
        rows.append({
            "threshold": threshold,
            "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "specificity": tn / (tn + fp) if tn + fp else np.nan,
            "PPV": tp / (tp + fp) if tp + fp else np.nan,
            "NPV": tn / (tn + fn) if tn + fn else np.nan,
            "flagged_pct": pred.mean(),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        })
    return pd.DataFrame(rows)

threshold_table = threshold_metrics(y, best_oof, [0.10, 0.15, 0.20, 0.25, 0.30, 0.40])
display(threshold_table.style.format({
    "threshold": "{:.2f}", "sensitivity": "{:.1%}", "specificity": "{:.1%}",
    "PPV": "{:.1%}", "NPV": "{:.1%}", "flagged_pct": "{:.1%}"
}))

## 6. Cross-validated permutation importance

For each held-out fold, one raw feature at a time is shuffled and the decrease in ROC AUC is measured. This is predictive importance—not causality. Correlated or redundant features can share importance, and negative values can arise from sampling noise.

In [ ]:
importance_frames = []
selected_model = models[best_model_name]
for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    fitted = clone(selected_model).fit(X.iloc[train_idx], y.iloc[train_idx])
    result = permutation_importance(
        fitted, X.iloc[test_idx], y.iloc[test_idx], scoring="roc_auc",
        n_repeats=8, random_state=RANDOM_STATE + fold, n_jobs=-1
    )
    importance_frames.append(pd.DataFrame({
        "feature": features,
        "importance": result.importances_mean,
        "fold": fold,
    }))

importance = pd.concat(importance_frames, ignore_index=True)
importance_summary = (importance.groupby("feature")["importance"]
                      .agg(["mean", "std"])
                      .sort_values("mean", ascending=False))
display(importance_summary.style.format("{:.3f}"))

plot_data = importance_summary.sort_values("mean")
plt.figure(figsize=(8, 5))
plt.barh(plot_data.index, plot_data["mean"], xerr=plot_data["std"], color="#54A24B", alpha=0.85)
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Mean held-out ROC AUC decrease when shuffled")
plt.title(f"Cross-validated permutation importance: {best_model_name}")
plt.tight_layout()
plt.show()

## 7. Calibration and subgroup diagnostics

Calibration is summarized in equal-count probability bins. The subgroup table is diagnostic only: it does not establish fairness, and smaller groups have more uncertainty. A deployment study should define protected groups and acceptable performance criteria in advance.

In [ ]:
diagnostic = data[["sex"]].copy()
diagnostic["outcome"] = y.to_numpy()
diagnostic["probability"] = best_oof
diagnostic["risk_bin"] = pd.qcut(diagnostic["probability"], q=10, duplicates="drop")

calibration = diagnostic.groupby("risk_bin", observed=True).agg(
    n=("outcome", "size"),
    mean_predicted=("probability", "mean"),
    observed_rate=("outcome", "mean"),
).reset_index(drop=True)
display(calibration.style.format({"mean_predicted": "{:.3f}", "observed_rate": "{:.3f}"}))

plt.figure(figsize=(6, 5))
plt.plot(calibration["mean_predicted"], calibration["observed_rate"], "o-", label=best_model_name)
plt.plot([0, 1], [0, 1], "--", color="grey", linewidth=1, label="Perfect calibration")
plt.xlim(0, max(0.45, calibration[["mean_predicted", "observed_rate"]].to_numpy().max() + 0.03))
plt.ylim(0, max(0.45, calibration[["mean_predicted", "observed_rate"]].to_numpy().max() + 0.03))
plt.xlabel("Mean predicted risk")
plt.ylabel("Observed readmission rate")
plt.title("Out-of-fold calibration by risk decile")
plt.legend()
plt.tight_layout()
plt.show()

subgroup_rows = []
for sex, frame in diagnostic.groupby("sex"):
    subgroup_rows.append({
        "sex": sex,
        "n": len(frame),
        "prevalence": frame["outcome"].mean(),
        "roc_auc": roc_auc_score(frame["outcome"], frame["probability"]),
        "average_precision": average_precision_score(frame["outcome"], frame["probability"]),
        "brier_score": brier_score_loss(frame["outcome"], frame["probability"]),
    })
display(pd.DataFrame(subgroup_rows).style.format({
    "prevalence": "{:.1%}", "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}", "brier_score": "{:.3f}"
}))

## 8. Conclusion and next steps

**Conclusion:** yes, these variables predict 30-day readmission better than a prevalence-only baseline, but only modestly. The result is most consistent with a **risk-ranking aid** for further prospective evaluation, not a clinically ready decision rule.

Important limitations:

1. There is no encounter date, hospital/site indicator, diagnosis, comorbidity, prior utilization window, discharge disposition, or social-risk information. Exact age and temporal validation are therefore impossible.
2. Grouped folds test generalization to unseen people, but all rows may still come from one institution and period. Performance can fall under site or time shift.
3. `visit_number` and discharge variables may be legitimate at discharge, but their operational timestamps and definitions must be verified. If prediction is intended earlier (for example, at admission), discharge-time fields must be removed.
4. A 30-day readmission label can be incomplete if readmissions outside the source system are not captured.
5. Geographic information can encode sensitive socioeconomic patterns. Fairness, access, and intervention impact require dedicated assessment.
6. The reported confidence intervals do not account for choosing the best model from this same cross-validation experiment.

Recommended next steps:

- define the exact decision time and intervention;
- add encounter dates and perform an out-of-time test, then validate at a separate site;
- confirm label completeness and feature provenance;
- compare against an accepted clinical score if one is applicable;
- select a threshold from costs/capacity on training data and lock it before final evaluation;
- assess calibration and performance across prespecified demographic and operational subgroups;
- run a prospective silent trial and evaluate whether acting on predictions improves patient outcomes.

In [ ]:
# Optional: fit the selected pipeline to all available data for method development.
# Do not deploy this object without external/temporal validation and a locked threshold.
final_model = clone(models[best_model_name]).fit(X, y)
print(f"Fitted final development model: {best_model_name}")